# Resample Datasets for Backbone Inspection
Create fixed, balanced sample CSVs for VinDr and EMBED that prioritize rare labels.

**Strategy (Greedy Set Cover):**
1. **Mandatory**: keep ALL rows containing at least one rare-class value across any clinical attribute.
2. **Assessment**: check how much quota each common class already gained for free from step 1.
3. **Greedy fill**: among remaining rows, iteratively pick those that cover the **most** unfilled common-class quotas simultaneously, minimizing total sample size while meeting all per-class caps.

In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

# ---- Paths ----
vindr_csv = os.environ.get('VINDR_BREAST_CSV', '/path/to/Mammo/VinDr-Mammo/breast-level_annotations.csv')
vindr_meta_csv = os.environ.get('VINDR_METADATA_CSV', '/path/to/Mammo/VinDr-Mammo/metadata.csv')
vindr_data_dir = os.environ.get('VINDR_DATA_DIR', '/path/to/Mammo/VinDr-Mammo')
embed_csv = 'data/embed_pretrain.csv'
embed_data_dir = os.environ.get('EMBED_IMAGE_DIR', '/path/to/Mammo/EMBED/pngs/1024x768')

output_dir = 'data/inspection_samples'
os.makedirs(output_dir, exist_ok=True)


In [ ]:
def greedy_priority_resample(df, clinical_attrs, cap_per_class=1000, batch_size=100):
    """Greedy set-cover resampling: keep all rare rows, then greedily fill common quotas.

    Instead of sampling each attribute independently (which wastes rows on overlap),
    this treats common-class quota filling as a set cover problem:
    - Step 1 (Mandatory): select every row that has at least one rare-class value.
    - Step 2 (Assessment): compute remaining quota per common class after step 1.
    - Step 3 (Greedy): repeatedly pick rows that satisfy the most outstanding quotas,
      so one row can count toward multiple attributes at once.

    Args:
        df: Full DataFrame (reset-indexed).
        clinical_attrs: dict mapping column -> {value: 'rare'|'common'}.
        cap_per_class: Target sample count per common class per attribute.
        batch_size: Rows to pick per greedy iteration (speed vs optimality).

    Returns:
        Resampled DataFrame (deduplicated, shuffled).
    """
    # --- Step 1: mandatory rare rows ---
    rare_mask = pd.Series(False, index=df.index)
    attr_classes = {}  # (attr, val) -> 'rare'|'common'

    for attr, class_spec in clinical_attrs.items():
        if attr not in df.columns:
            print(f'  Skipping {attr} (not in data)')
            continue
        for val, label in class_spec.items():
            attr_classes[(attr, val)] = label
            if label == 'rare':
                rare_mask |= (df[attr] == val)

    sampled = set(df.index[rare_mask].tolist())
    print(f'  Mandatory (rare): {len(sampled)} rows')

    # --- Step 2: assess remaining common quotas ---
    requirements = {}  # (attr, val) -> needed count
    for (attr, val), label in attr_classes.items():
        if label != 'common':
            continue
        current = sum(1 for i in sampled if df.at[i, attr] == val)
        needed = max(0, cap_per_class - current)
        if needed > 0:
            requirements[(attr, val)] = needed
            print(f'  {attr}={val}: have {current}/{cap_per_class}, need {needed} more')
        else:
            print(f'  {attr}={val}: already satisfied ({current}>={cap_per_class})')

    # --- Step 3: greedy fill ---
    remaining_idx = np.array([i for i in df.index if i not in sampled])

    iteration = 0
    while requirements and len(remaining_idx) > 0:
        # Score each remaining row: how many open requirements does it help?
        scores = np.zeros(len(remaining_idx), dtype=int)
        for (attr, val) in requirements:
            mask = df.loc[remaining_idx, attr].values == val
            scores += mask.astype(int)

        if scores.max() == 0:
            break

        # Pick top-scoring rows (batch)
        top_k = min(batch_size, (scores > 0).sum())
        top_positions = np.argpartition(-scores, top_k)[:top_k]
        # Among top_k, pick those with highest scores first (for better coverage)
        top_positions = top_positions[np.argsort(-scores[top_positions])]
        chosen = remaining_idx[top_positions]

        # Add chosen rows and update quotas
        for idx in chosen:
            sampled.add(idx)
            for (attr, val) in list(requirements.keys()):
                if df.at[idx, attr] == val:
                    requirements[(attr, val)] -= 1
                    if requirements[(attr, val)] <= 0:
                        del requirements[(attr, val)]
            if not requirements:
                break

        remaining_idx = np.array([i for i in remaining_idx if i not in sampled])
        iteration += 1

    print(f'  Greedy done in {iteration} iterations')

    result = df.loc[sorted(sampled)].sample(frac=1.0, random_state=42).reset_index(drop=True)
    return result

## VinDr-Mammo
- **breast_birads**: BI-RADS 3, 4, 5 are rare -> keep all. Cap BI-RADS 1/2 at 1000.
- **breast_density**: DENSITY A is rare -> keep all. Cap B/C/D at 1000.
- **ManufacturerGroup**: normalize IMS vendor names into one group (`IMS`), keep all IMS rows, and cap SIEMENS/Planmed at 1000.


In [ ]:
df_vindr = pd.read_csv(vindr_csv)
df_vindr['study_id'] = df_vindr['study_id'].astype(str)
df_vindr['image_id'] = df_vindr['image_id'].astype(str)

# Add scanner metadata for acquisition-domain inspection/resampling.
df_vindr_meta = pd.read_csv(
    vindr_meta_csv,
    usecols=['SOP Instance UID', 'Manufacturer'],
).rename(columns={'SOP Instance UID': 'image_id'})
df_vindr_meta['image_id'] = df_vindr_meta['image_id'].astype(str)
df_vindr = df_vindr.merge(df_vindr_meta, on='image_id', how='left', validate='m:1')

df_vindr['ManufacturerGroup'] = df_vindr['Manufacturer'].replace({
    'IMS s.r.l.': 'IMS',
    'IMS GIOTTO S.p.A.': 'IMS',
})

# Build img_path (same logic as inspection notebook)
df_vindr['img_path'] = [
    os.path.join(vindr_data_dir, 'pngs/1024x768', str(sid), str(iid) + '.png')
    for sid, iid in zip(df_vindr.study_id, df_vindr.image_id)
]

# Drop rows without birads label
df_vindr = df_vindr[df_vindr['breast_birads'].notna()].reset_index(drop=True)

print(f'VinDr full: {len(df_vindr)} rows')
print(f'\nbreast_birads:\n{df_vindr.breast_birads.value_counts().sort_index()}')
print(f'\nbreast_density:\n{df_vindr.breast_density.value_counts().sort_index()}')
print(f'\nManufacturer:\n{df_vindr.Manufacturer.value_counts(dropna=False)}')
print(f'\nManufacturerGroup:\n{df_vindr.ManufacturerGroup.value_counts(dropna=False)}')


In [ ]:
vindr_clinical = {
    'breast_birads': {
        'BI-RADS 3': 'rare',
        'BI-RADS 4': 'rare',
        'BI-RADS 5': 'rare',
        'BI-RADS 1': 'common',
        'BI-RADS 2': 'common',
    },
    'breast_density': {
        'DENSITY A': 'rare',
        'DENSITY B': 'common',
        'DENSITY C': 'common',
        'DENSITY D': 'common',
    },
    'ManufacturerGroup': {
        'IMS': 'rare',
        'SIEMENS': 'common',
        'Planmed': 'common',
    },
}

print('Resampling VinDr (greedy set cover)...')
df_vindr_sample = greedy_priority_resample(df_vindr, vindr_clinical, cap_per_class=1000)

print(f'\n=== VinDr sample: {len(df_vindr_sample)} rows ===')
print(f'\nbreast_birads:\n{df_vindr_sample.breast_birads.value_counts().sort_index()}')
print(f'\nbreast_density:\n{df_vindr_sample.breast_density.value_counts().sort_index()}')
print(f'\nManufacturer:\n{df_vindr_sample.Manufacturer.value_counts(dropna=False)}')
print(f'\nManufacturerGroup:\n{df_vindr_sample.ManufacturerGroup.value_counts(dropna=False)}')
print(f'\nview_position:\n{df_vindr_sample.view_position.value_counts()}')
print(f'\nlaterality:\n{df_vindr_sample.laterality.value_counts()}')


## EMBED
Using `data/embed_pretrain.csv`.
- **asses / breast_birads**: keep BI-RADS 1-5 only (`N`, `B`, `P`, `S`, `M`) so plots are comparable with VinDr. Drop BI-RADS 0 (`A`), BI-RADS 6 (`K`), and X/unmapped assessments.
- **asses / breast_birads**: N, B, P, S are common enough -> cap at 2000. M is rare -> keep all.
- **tissueden** (density): all 1/2/3/4 are common enough -> cap each at 2000.
- **ManufacturerGroup**: normalize GE vendor names into one group (`GE`). Cap HOLOGIC/GE/FUJIFILM at 2000.


In [ ]:
df_embed = pd.read_csv(embed_csv, low_memory=False)

# Standard filters (same as inspection notebook)
df_embed = df_embed[df_embed['FinalImageType'] == '2D']
df_embed = df_embed[df_embed['GENDER_DESC'] == 'Female']
df_embed = df_embed[df_embed['tissueden'].notna()]
df_embed = df_embed[df_embed['tissueden'] < 5]
df_embed = df_embed[df_embed['ViewPosition'].isin(['MLO', 'CC'])]
df_embed = df_embed[df_embed['spot_mag'].isna()]

# Build img_path
df_embed['img_path'] = [os.path.join(embed_data_dir, p) for p in df_embed.image_path.values]
df_embed['study_id'] = df_embed['empi_anon'].astype(str)
df_embed['image_id'] = [p.split('/')[-1] for p in df_embed.image_path.values]
df_embed['laterality'] = df_embed['ImageLateralityFinal']
df_embed['ManufacturerGroup'] = df_embed['Manufacturer'].replace({
    'GE MEDICAL SYSTEMS': 'GE',
    'GE HEALTHCARE': 'GE',
    'HOLOGIC, Inc.': 'HOLOGIC',
    'FUJIFILM Corporation': 'FUJIFILM',
})

# Derived columns
if 'density' not in df_embed.columns:
    df_embed['density'] = df_embed['tissueden'].map({1: 'A', 2: 'B', 3: 'C', 4: 'D'})

embed_birads_map = {
    'N': 'BI-RADS 1 (N)',
    'B': 'BI-RADS 2 (B)',
    'P': 'BI-RADS 3 (P)',
    'S': 'BI-RADS 4 (S)',
    'M': 'BI-RADS 5 (M)',
}
df_embed['breast_birads'] = df_embed['asses'].map(embed_birads_map)
df_embed = df_embed[df_embed['breast_birads'].notna()]

df_embed = df_embed.reset_index(drop=True)

print(f'EMBED filtered: {len(df_embed)} rows')
print(f'\nasses:\n{df_embed.asses.value_counts().sort_index()}')
print(f'\nbreast_birads:\n{df_embed.breast_birads.value_counts().sort_index()}')
print(f'\ntissueden:\n{df_embed.tissueden.value_counts().sort_index()}')
print(f'\nManufacturer:\n{df_embed.Manufacturer.value_counts(dropna=False)}')
print(f'\nManufacturerGroup:\n{df_embed.ManufacturerGroup.value_counts(dropna=False)}')


In [ ]:
embed_clinical = {
    'asses': {
        'N': 'common',
        'B': 'common',
        'P': 'common',
        'S': 'common',
        'M': 'rare',
    },
    'tissueden': {
        1.0: 'common',
        2.0: 'common',
        3.0: 'common',
        4.0: 'common',
    },
    'ManufacturerGroup': {
        'HOLOGIC': 'common',
        'GE': 'common',
        'FUJIFILM': 'common',
    },
}

print('Resampling EMBED (greedy set cover)...')
df_embed_sample = greedy_priority_resample(df_embed, embed_clinical, cap_per_class=2000)

print(f'\n=== EMBED sample: {len(df_embed_sample)} rows ===')
print(f'\nasses:\n{df_embed_sample.asses.value_counts().sort_index()}')
print(f'\nbreast_birads:\n{df_embed_sample.breast_birads.value_counts().sort_index()}')
print(f'\ntissueden:\n{df_embed_sample.tissueden.value_counts().sort_index()}')
print(f'\nViewPosition:\n{df_embed_sample.ViewPosition.value_counts()}')
print(f'\nImageLateralityFinal:\n{df_embed_sample.ImageLateralityFinal.value_counts()}')
print(f'\nManufacturer:\n{df_embed_sample.Manufacturer.value_counts(dropna=False)}')
print(f'\nManufacturerGroup:\n{df_embed_sample.ManufacturerGroup.value_counts(dropna=False)}')


## Save sample CSVs

In [ ]:
vindr_out = os.path.join(output_dir, 'vindr_inspection_sample.csv')
embed_out = os.path.join(output_dir, 'embed_inspection_sample.csv')

# Keep vendor-level columns only; scanner-model labels are intentionally not saved.
df_vindr_sample = df_vindr_sample.drop(columns=['ManufacturerModelName'], errors='ignore')
df_embed_sample = df_embed_sample.drop(columns=['ManufacturerModelName'], errors='ignore')

df_vindr_sample.to_csv(vindr_out, index=False)
df_embed_sample.to_csv(embed_out, index=False)

print(f'Saved: {vindr_out}  ({len(df_vindr_sample)} rows)')
print(f'Saved: {embed_out}  ({len(df_embed_sample)} rows)')
